# 00 — Code Walkthrough (v3, DAD IoT-network)

A guided tour of **every module in `src/`**, in the order data flows through them, each with a short
explanation and a runnable example. This is the fastest way to understand the whole system.

```
 preprocess ─► features ─► channels ─► fusion ─► evaluation      (find attacks, real labels)
      │                                                │
      └─► query_engine ─► interface_parsing ─► interface ─► llm   (answer questions, verifiably)
```
The **one principle**: deterministic code computes every number (the *truth layer*); the LLM only
explains, and a faithfulness check verifies it. Run **Restart & Run All** (after `bash tools/get_dad_dataset.sh`).

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (9, 3); plt.rcParams["axes.grid"] = True
print("environment ready")

environment ready


## 1. `preprocess` — packets → a labelled table

DAD ships raw **`.pcap`** network captures (MQTT traffic) plus per-packet attack labels. `preprocess`
parses each capture with **scapy** (the temperature is a plain-float MQTT PUBLISH payload) and joins the
labels on `frame.number` (which equals the packet index) into one datetime-indexed **MESSAGES** frame —
the single source of truth for everything else.

In [2]:
from src import preprocess
df = preprocess.load_week()
print("MESSAGES frame:", df.shape, "| attacks:", int(df.label.sum()), "| readings:", int(df.temperature.notna().sum()))
df.loc[df.label == 1, ["src_ip", "clientid", "temperature", "attack_type", "label"]].head(3)

MESSAGES frame: (101583, 10) | attacks: 544 | readings: 32288


,src_ip,clientid,temperature,attack_type,label
ts,,,,,
2019-10-23 02:10:07.155519962,10.6.56.41,133.0,18.02,modification,1
2019-10-23 02:10:07.155544996,10.6.56.41,131.0,9.96,modification,1
2019-10-23 02:10:07.155767918,10.6.56.41,132.0,13.89,modification,1


## 2. `features` — two detector channels

The attacks reveal themselves in two different signals, so we build two feature families on a 5-minute
grid (the sensor publish cadence):
- **`sensor_series`** — temperature per sensor (the **value** channel; catches *modification*). With
  `fill=True` the 0.6% gaps are interpolated (value channel only — flow gaps are the interception signal).
- **`flow_features`** — behavioural traffic features: message rate, inter-arrival, duplicate-msgid ratio
  (the **flow** channel; catches *duplication/interception*). Attacker-IP identity is *deliberately
  excluded* to avoid label leakage.
- **`window_labels`** — the ground-truth label + attack type per window.

In [3]:
from src import features
sensors = features.sensor_series(df, "5min", fill=True)
flow    = features.flow_features(df, "5min")
labels  = features.window_labels(df, "5min")
print("value grid :", sensors.shape, "(windows x sensors)")
print("flow feats :", list(flow.columns))
print("windows    :", len(labels), "| attack windows:", int(labels.label.sum()))
flow.head(3)

value grid : (2016, 16) (windows x sensors)
flow feats : ['msg_rate', 'publish_rate', 'iat_mean', 'iat_std', 'dup_msgid_ratio']
windows    : 2016 | attack windows: 90


,msg_rate,publish_rate,iat_mean,iat_std,dup_msgid_ratio
ts,,,,,
2019-10-20 22:00:00,162.0,16.0,0.100248,1.256106,0.0
2019-10-20 22:05:00,50.0,16.0,11.150594,55.185545,0.0
2019-10-20 22:10:00,48.0,16.0,0.556104,3.844188,0.0


## 3. `channels` — one anomaly score per channel
- **`value_score`** = the largest |z-score| of temperature across sensors in each window (a modified
  reading spikes its sensor's z).
- **`flow_score`** = an **Isolation Forest** anomaly score over the flow features (higher = more unusual).

In [4]:
from src import channels
value_s = channels.value_score(sensors.reindex(labels.index))
flow_s  = channels.flow_score(flow.reindex(labels.index).fillna(0.0))
print("value_score range:", round(value_s.min(),2), "→", round(value_s.max(),2))
print("flow_score  range:", round(flow_s.min(),2), "→", round(flow_s.max(),2))

value_score range: 0.21 → 6.97
flow_score  range: 0.36 → 0.8


## 4. `fusion` — combine the two channels
- **`late_fusion`** — rank-normalise each score and take the max (an OR of two specialists).
- **`early_fusion`** — concatenate the feature sets → one Isolation Forest.
- **`supervised_scores`** — a logistic regression over the two scores, honestly evaluated with
  **day-blocked** cross-validation.

In [5]:
from src import fusion
late = fusion.late_fusion(value_s, flow_s, rule="max")
print("late-fusion score range:", round(late.min(),2), "→", round(late.max(),2))
print("(early_fusion and supervised_scores are demonstrated in 02_methods_and_evaluation)")

late-fusion score range: 0.02 → 1.0
(early_fusion and supervised_scores are demonstrated in 02_methods_and_evaluation)


## 5. `evaluation` — honest metrics for a rare, labelled problem
Because attacks are rare (~4.5% of windows), we use **average precision** (threshold-free),
**best-F1**, **point-adjusted** event-level recall, and **bootstrap confidence intervals** — never plain
accuracy.

In [6]:
from src import evaluation as ev
y = labels.label.to_numpy()
print("value channel  AP:", round(ev.average_precision(value_s.to_numpy(), y), 3))
print("flow  channel  AP:", round(ev.average_precision(flow_s.to_numpy(), y), 3))
print("best-F1 (value)  :", {k: round(v,3) for k,v in ev.best_f1(value_s.to_numpy(), y).items()})
print("AP 95% CI (value):", tuple(round(x,3) for x in ev.bootstrap_ci(value_s.to_numpy(), y)))

value channel  AP: 0.183
flow  channel  AP: 0.136
best-F1 (value)  : {'f1': 0.379, 'precision': 0.331, 'recall': 0.444, 'threshold': 3.798}
AP 95% CI (value): (0.136, 0.245)


## 6. `query_engine` — the truth layer
Every question's number is one **exact** pandas computation here — never produced by a model. This is
what makes answers verifiable (and independently re-checked by `verify_results.py`, 12/12).

In [7]:
from src import query_engine as qe
print("total attacks :", qe.count_attacks(df))
print("by day        :", qe.attacks_by_day(df))
print("by type       :", qe.attack_breakdown(df))
print("attacker IP   :", qe.attacker_ips(df))
print("sensors       :", len(qe.sensor_ids(df)))
print("hottest Friday:", round(qe.temperature_stat(df, "max", day="fri"), 2), "°C")

total attacks : 544
by day        : {'mon': 0, 'tue': 0, 'wed': 104, 'thu': 56, 'fri': 192, 'sat': 192, 'sun': 0}
by type       : {'duplication': 56, 'mixed': 384, 'modification': 104}
attacker IP   : ['10.6.56.41']
sensors       : 16
hottest Friday: 29.28 °C


## 7. `interface_parsing` + `interface` — plain-English → verified answer
A **deterministic** parser maps a question to an intent (count, attacker, temperature, sensors,
anomalies, …) and scope; `interface` calls the truth layer and returns a **trust-labelled** `Answer`
(🔢 computed / 📚 context / 🧠 inferred). Unknown questions are *refused, not guessed*.

In [8]:
from src import interface
bot = interface.Interface(df)
for q in ["How many attacks on Wednesday?", "Show all the sensors in the dataset",
          "Show me the anomalies", "flibbertigibbet"]:
    a = bot.ask(q)
    print(f"Q: {q}\n   → {a.text}")

Q: How many attacks on Wednesday?
   → There were 104 labelled attack packets on Wednesday.
Q: Show all the sensors in the dataset
   → The dataset has 16 sensors (MQTT client ids): [131, 132, 133, 134, 151, 152, 153, 154, 231, 232, 233, 234, 251, 252, 253, 254].
Q: Show me the anomalies
   → There are 544 labelled anomalies (network attacks) across 4 days: Wednesday, Thursday, Friday, Saturday.
Q: flibbertigibbet
   → I couldn't map that to a known query. Try asking about attacks, attack types, the attacker IP, temperatures, or traffic.


## 8. `llm` — optional local explanation + the faithfulness gate
The LLM only *rewords* the computed facts. A **faithfulness check** compares the numbers in the facts vs
the wording and rejects any invented figure — so even a small local model is safe.

In [9]:
from src import llm
facts = "There were 104 labelled attack packets on Wednesday."
print("good  :", llm.faithfulness(facts, "On Wednesday, 104 attack packets were flagged."))
print("bad   :", llm.faithfulness(facts, "There were roughly 900 attacks."))
print("Ollama available:", llm.is_available())

good  : {'ok': True, 'dropped': [], 'invented': []}
bad   : {'ok': False, 'dropped': [104.0], 'invented': [900.0]}
Ollama available: True


## 9. `experiment` — the end-to-end evaluation runner
`run_experiment` ties it together: builds the windows, computes both channel scores, forms every fusion,
and grades all configs on the real labels (window-level + point-adjusted + CIs + per attack type). This
is the headline result — explored in depth in **02_methods_and_evaluation**.

In [10]:
from src import experiment
res = experiment.run_experiment(df)
res[["config", "AP", "f1", "recall_pa", "recall_modification", "recall_duplication"]].round(3)

,config,AP,f1,recall_pa,recall_modification,recall_duplication
0,flow_only,0.136,0.226,0.178,0.000,1.0
1,value_only,0.183,0.379,0.889,0.462,0.5
2,early_fusion,0.147,0.289,1.000,0.423,1.0
3,late_fusion,0.171,0.306,1.000,0.462,1.0
4,supervised,0.101,0.212,1.000,0.423,1.0


## Where to go next
- **01_data_exploration_and_visualization** — the dataset in depth (distributions, STL trend/seasonality, correlations, attacks).
- **02_methods_and_evaluation** — parsing correctness, the detectors, the fusion experiment, the metrics.
- **03_interface_and_explanation** — the NL interface and the LLM faithfulness gate.
- **04_build_along_tutorial** — a from-scratch teaching walkthrough.